In this notebook, we try : $\newline$
##### Baseline:
- ROCKET
- Multi-ROCKET
- (TS2VEC)

##### Foundation Model:
- MANTIS
- CHRONOS
- TimesFM

In [2]:
!pip install tslearn sktime scikit-learn mantis-tsfm aeon chronos-forecasting ts2vec

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 387.9/387.9 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.7/72.7 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.8/434.8 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.8/159.8 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 87.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/

In [3]:
!pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 616.3/616.3 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 46.3 MB/s eta 0:00:00
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.2
    Uninstalling hf-xet-1.3.2:
      Successfully uninstalled hf-xet-1.3.2
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.6
    Uninstalling transformers-4.57.6:
      Successfully uninstalled transformers-4.57.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
chronos-forecasting 2.2.2 requires transformers<5,>=4.41, but you 

In [4]:
import numpy as np
import pandas as pd
from IPython.display import display
import time
import torch

from aeon.transformations.collection.convolution_based import MultiRocket
from chronos import ChronosPipeline
from mantis.architecture import Mantis8M
from mantis.trainer import MantisTrainer
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sktime.classification.kernel_based import RocketClassifier
from sklearn.impute import SimpleImputer
import torch.nn.functional as F
from transformers import AutoModel
from tslearn.datasets import UCR_UEA_datasets
from ts2vec import TS2Vec


In [5]:
print("Loading dataset LSST...")
ds = UCR_UEA_datasets()
X_train_raw, y_train_raw, X_test_raw, y_test_raw = ds.load_dataset("LSST")

# (N, Timesteps, Channels) -> (N, Channels, Timesteps)
X_train = X_train_raw.swapaxes(1, 2)
X_test = X_test_raw.swapaxes(1, 2)

def impute_channels(X):
    n, c, t = X.shape
    X_flat = X.reshape(n * c, t)
    imputer = SimpleImputer(strategy='mean')
    X_flat = imputer.fit_transform(X_flat)
    return X_flat.reshape(n, c, t)

X_train = impute_channels(X_train)
X_test  = impute_channels(X_test)

# Encoding labels
le = LabelEncoder()
y_train = le.fit_transform(y_train_raw)
y_test  = le.transform(y_test_raw)

print(f"Train shape : {X_train.shape}")
print(f"Test shape  : {X_test.shape}")

Loading dataset LSST...
Train shape : (2459, 6, 36)
Test shape  : (2466, 6, 36)


# Baseline

## Rocket

In [9]:
print("--- Baseline : Rocket ---")
clf_rocket = RocketClassifier(num_kernels=10000, random_state=42)

# Training Time
start_train = time.time()
clf_rocket.fit(X_train, y_train)
end_train = time.time()

# Inference time
start_test = time.time()
preds_rocket = clf_rocket.predict(X_test)
end_test = time.time()
time_train_rocket = end_train - start_train
time_test_rocket = end_test - start_test

print(f"Training Time : {time_train_rocket:.2f} sec")
print(f"Inference Time : {time_train_rocket:.2f} sec")

--- Baseline : Rocket ---
Training Time : 167.41 sec
Inference Time : 167.41 sec


In [10]:
acc_rocket = accuracy_score(y_test, preds_rocket)
f1_weighted_rocket = f1_score(y_test, preds_rocket, average='weighted')
f1_macro_rocket = f1_score(y_test, preds_rocket, average='macro')
f = f1_score(y_test, preds_rocket, average=None)

print(f"F1-Score (Weighted) ROCKET  : {f1_weighted_rocket * 100:.2f}%")
print(f"F1-Score (Macro) ROCKET  : {f1_macro_rocket * 100:.2f}%")

F1-Score (Weighted) ROCKET  : 60.28%
F1-Score (Macro) ROCKET  : 40.07%


## Multi Rocket

In [11]:
print("--- Baseline : MultiROCKET ---")
model_mr = MultiRocket(n_kernels=10000, random_state=42)

clf_lr_mr = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42)
)

# Training time
start_train = time.time()
Z_train_mr = model_mr.fit_transform(X_train)
clf_lr_mr.fit(Z_train_mr, y_train)
end_train = time.time()

# Inference time
start_test = time.time()
Z_test_mr = model_mr.transform(X_test)
preds_multirocket = clf_lr_mr.predict(Z_test_mr)
end_test = time.time()

time_train_multirocket = end_train - start_train
time_test_multirocket = end_test - start_test

print(f"Training Time : {time_train_multirocket:.2f} sec")
print(f"Inference Time : {time_test_multirocket:.2f} sec")

--- Baseline : MultiROCKET ---
Training Time : 110.21 sec
Inference Time : 9.92 sec


In [12]:
# Scores
f1_macro_multirocket = f1_score(y_test, preds_multirocket, average='weighted', zero_division=0)
f1_weighted_multirocket = f1_score(y_test, preds_multirocket, average="macro", zero_division=0)

# Results
print(f"F1-Score (Weighted) MultiROCKET : {f1_weighted_multirocket * 100:.2f}%")
print(f"F1-Score (Avg) MultiROCKET : {f1_macro_multirocket * 100:.2f}%")


F1-Score (Weighted) MultiROCKET : 53.06%
F1-Score (Avg) MultiROCKET : 63.84%


## Adapting a Foundation Model

In [13]:
# GPU for colab
device = "cuda" if torch.cuda.is_available() else "cpu"

## Resize of the datasets

In [7]:
def resize_for_mantis(X, target_length=512):
    # X shape = (N, Channels, Timesteps)
    X_tensor = torch.tensor(X, dtype=torch.float)
    X_scaled = F.interpolate(X_tensor, size=target_length, mode='linear', align_corners=False)
    return X_scaled.numpy()

## MANTIS : Adaptation 'Head' (Linear Probing)

In [14]:
# Foundation models loading
network = Mantis8M(device=device)
network = network.from_pretrained("paris-noah/Mantis-8M")
model_mantis_head = MantisTrainer(network=network, device=device)

# Resizing data
X_train_mantis = resize_for_mantis(X_train)
X_test_mantis = resize_for_mantis(X_test)

# Training Time
start_train = time.time()
model_mantis_head.fit(
    X_train_mantis,
    y_train,
    num_epochs=100,
    fine_tuning_type="head"
)
end_train = time.time()

# Inference Time
start_test = time.time()
preds_head = model_mantis_head.predict(X_test_mantis)
end_test = time.time()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/335 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/32.5M [00:00<?, ?B/s]

Epoch 99: Train Loss 0.6499: 100%|██████████| 100/100 [00:03<00:00, 28.17it/s]


In [15]:
time_train_head = end_train - start_train
time_test_head = end_test - start_test
f1_weighted_head = f1_score(y_test, preds_head, average='weighted', zero_division=0)
f1_macro_head = f1_score(y_test, preds_head, average='macro', zero_division=0)
report_head = classification_report(y_test, preds_head, zero_division=0)

print("Results : MANTIS (Adaptation 'Head' / Linear Probing)")
print(f"F1-Score (Weighted)  : {f1_weighted_head * 100:.2f}%")
print(f"F1-Score (Macro)     : {f1_macro_head * 100:.2f}%")
print(f"Training Time        : {time_train_head:.2f} sec")
print(f"Inference Time       : {time_test_head:.2f} sec")

Results : MANTIS (Adaptation 'Head' / Linear Probing)
F1-Score (Weighted)  : 63.54%
F1-Score (Macro)     : 51.15%
Training Time        : 8.30 sec
Inference Time       : 4.00 sec


## MANTIS : Adaptation 'Full' (Full Fine-Tuning)

In [16]:
model_mantis_full = MantisTrainer(network=network, device=device)

# Training Time
start_train = time.time()
model_mantis_full.fit(
    X_train_mantis,
    y_train,
    num_epochs=50,
    fine_tuning_type="full"
)
end_train = time.time()
time_train_full = end_train - start_train

# Inference time
start_test = time.time()
preds_full = model_mantis_full.predict(X_test_mantis)
end_test = time.time()
time_test_full = end_test - start_test


print(f"Training Time : {time_train_full:.2f} sec")
print(f"Inference Time : {time_test_full:.2f} sec")

Epoch 49: Train Loss 0.0537: 100%|██████████| 50/50 [11:22<00:00, 13.64s/it]


Training Time : 682.10 sec
Inference Time : 4.93 sec


In [17]:
# Time and scores
f1_weighted_full = f1_score(y_test, preds_full, average='weighted')
f1_macro_full = f1_score(y_test, preds_full, average='macro')
report_full = classification_report(y_test, preds_full)

print("=== Results : MANTIS (Full Fine-Tuning) ===")
print(f"F1-Score (Weighted)  : {f1_weighted_full * 100:.2f}%")
print(f"F1-Score (Macro)     : {f1_macro_full  * 100:.2f}%")
print(f"Training Time        : {time_train_full:.2f} sec")
print(f"Inference Time       : {time_test_full:.2f} sec")

=== Results : MANTIS (Full Fine-Tuning) ===
F1-Score (Weighted)  : 68.37%
F1-Score (Macro)     : 56.97%
Training Time        : 682.10 sec
Inference Time       : 4.93 sec


## Chronos Amazon

In [18]:
print("--- Initialisation of Chronos (Amazon) ---")
pipeline = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-mini",
    device_map=device,
    torch_dtype=torch.bfloat16
)

# Getting the Chronos embeddings
def get_chronos_embeddings(X_data, pipeline, batch_size=32):
    if len(X_data.shape) == 3:
        X_data_flat = X_data.reshape(X_data.shape[0], -1)
    else:
        X_data_flat = X_data

    embeddings_list = []

    with torch.no_grad():
        for i in range(0, len(X_data_flat), batch_size):
            batch = torch.tensor(X_data_flat[i:i+batch_size], dtype=torch.float32)
            batch_embeds, tokenizer_state = pipeline.embed(batch)
            batch_embeds_mean = batch_embeds.mean(dim=1)
            embeddings_list.append(batch_embeds_mean.cpu().float().numpy())

    return np.vstack(embeddings_list)

print("--- Feature Extraction ---")
start_train = time.time()
Z_train_chronos = get_chronos_embeddings(X_train, pipeline)

clf_lr_chronos = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42)
)
clf_lr_chronos.fit(Z_train_chronos, y_train)
end_train = time.time()

print("--- Test Inference ---")
start_test = time.time()
Z_test_chronos = get_chronos_embeddings(X_test, pipeline)
preds_chronos = clf_lr_chronos.predict(Z_test_chronos)
end_test = time.time()

time_train_chronos = end_train - start_train
time_test_chronos = end_test - start_test

--- Initialisation of Chronos (Amazon) ---


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/81.8M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/89 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

--- Feature Extraction ---
--- Test Inference ---


In [19]:
# --- Results ---
f1_chronos = f1_score(y_test, preds_chronos, average='macro', zero_division=0)

print(f"\n=== Results : CHRONOS (Feature Extraction + LR) ===")
print(f"F1-Score (Macro) : {f1_chronos * 100:.2f}%")
print(f"Training Time    : {time_train_chronos:.2f} sec")
print(f"Inference Time   : {time_test_chronos:.2f} sec")


=== Results : CHRONOS (Feature Extraction + LR) ===
F1-Score (Macro) : 50.16%
Training Time    : 15.34 sec
Inference Time   : 5.00 sec


## Google TimesFM

In [20]:
print("--- TimesFM: Feature Extraction + LR ---")
timesfm_model = AutoModel.from_pretrained(
    "google/timesfm-2.5-200m-transformers",
    trust_remote_code=True
).to(device)
timesfm_model.eval()

def get_timesfm_embeddings(X_data, model, batch_size=32, patch_len=32):
    embeddings = []

    with torch.no_grad():
        for i in range(0, len(X_data), batch_size):
            batch = torch.tensor(X_data[i:i+batch_size], dtype=torch.float32).to(device)
            b_size, n_channels, n_timesteps = batch.shape

            # Pad sequence length to nearest multiple of patch_len
            remainder = n_timesteps % patch_len
            if remainder != 0:
                pad_len = patch_len - remainder
                batch = torch.nn.functional.pad(batch, (0, pad_len))

            batch_reshaped = batch.reshape(b_size * n_channels, -1)
            outputs = model(past_values=batch_reshaped, output_hidden_states=True)

            pooled_time = outputs.last_hidden_state.mean(dim=1)
            final_embeds = pooled_time.reshape(b_size, n_channels, -1).mean(dim=1)

            embeddings.append(final_embeds.cpu().numpy())

    return np.vstack(embeddings)

clf_lr_timesfm = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))

start_train = time.time()
Z_train_timesfm = get_timesfm_embeddings(X_train, timesfm_model)
clf_lr_timesfm.fit(Z_train_timesfm, y_train)
time_train_timesfm = time.time() - start_train

start_test = time.time()
Z_test_timesfm = get_timesfm_embeddings(X_test, timesfm_model)
preds_timesfm = clf_lr_timesfm.predict(Z_test_timesfm)
time_test_timesfm = time.time() - start_test


--- TimesFM: Feature Extraction + LR ---


config.json:   0%|          | 0.00/914 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/925M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/266 [00:00<?, ?it/s]

TimesFm2_5Model LOAD REPORT from: google/timesfm-2.5-200m-transformers
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
output_projection_quantiles.residual_layer.weight | UNEXPECTED |  | 
output_projection_point.residual_layer.weight     | UNEXPECTED |  | 
output_projection_quantiles.input_layer.weight    | UNEXPECTED |  | 
output_projection_point.output_layer.weight       | UNEXPECTED |  | 
output_projection_point.input_layer.weight        | UNEXPECTED |  | 
output_projection_quantiles.output_layer.weight   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [21]:
f1_macro_TimesFM = f1_score(y_test, preds_timesfm, average='macro', zero_division=0)
f1_weighted_TimesFM = f1_score(y_test, preds_timesfm, average='weighted', zero_division=0)

print(f"F1-Score (Weighted) TimesFM: {f1_weighted_TimesFM * 100:.2f}%")
print(f"F1-Score (Macro) TimesFM    : {f1_macro_TimesFM * 100:.2f}%")

F1-Score (Weighted) TimesFM: 51.08%
F1-Score (Macro) TimesFM    : 36.10%


In [22]:
print("--- TS2Vec: Feature Extraction + LR ---")


# TS2Vec needs (N, Timesteps, Channels) → (N, C, T)
X_train_ts2vec = X_train.transpose(0, 2, 1)  # (2459, 36, 6)
X_test_ts2vec  = X_test.transpose(0, 2, 1)   # (2466, 36, 6)

# Training
ts2vec_model = TS2Vec(
    input_dims=X_train_ts2vec.shape[2],
    device=0 if torch.cuda.is_available() else -1,
    output_dims=320,
    batch_size=64,
    lr=0.001,
)

start_train = time.time()
ts2vec_model.fit(X_train_ts2vec, n_epochs=50, verbose=True)

# Embedding Extraction
Z_train_ts2vec = ts2vec_model.encode(X_train_ts2vec, encoding_window='full_series')

clf_lr_ts2vec = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=100, random_state=42)
)
clf_lr_ts2vec.fit(Z_train_ts2vec, y_train)
time_train_ts2vec = time.time() - start_train

start_test = time.time()
Z_test_ts2vec = ts2vec_model.encode(X_test_ts2vec, encoding_window='full_series')
preds_ts2vec  = clf_lr_ts2vec.predict(Z_test_ts2vec)
time_test_ts2vec = time.time() - start_test

--- TS2Vec: Feature Extraction + LR ---
Epoch #0: loss=1371899.6611842106
Epoch #1: loss=198089.96221602592
Epoch #2: loss=188067.36607601767
Epoch #3: loss=107098.29506964433
Epoch #4: loss=134036.18805252877
Epoch #5: loss=70031.91017231188
Epoch #6: loss=48157.06068661338
Epoch #7: loss=47033.23292300576
Epoch #8: loss=51479.76536800987
Epoch #9: loss=74310.93721731086
Epoch #10: loss=47246.11655787418
Epoch #11: loss=34303.83463648746
Epoch #12: loss=18795.66018516139
Epoch #13: loss=16943.788779810857
Epoch #14: loss=21950.85276151958
Epoch #15: loss=48972.99848375822
Epoch #16: loss=37506.9660403603
Epoch #17: loss=39693.113523784436
Epoch #18: loss=28869.784066451222
Epoch #19: loss=19955.349546733654
Epoch #20: loss=24829.385621723377
Epoch #21: loss=27021.24783646433
Epoch #22: loss=23662.620632773953
Epoch #23: loss=9924.375739649722
Epoch #24: loss=10306.28225547389
Epoch #25: loss=10779.112531561601
Epoch #26: loss=25456.22569033974
Epoch #27: loss=28464.563566509045
Epoch 

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [23]:
print(f"Training Time  : {time_train_ts2vec:.2f} sec")
print(f"Inference Time : {time_test_ts2vec:.2f} sec")
f1_weighted_ts2vec = f1_score(y_test, preds_ts2vec, average='weighted', zero_division=0)
f1_macro_ts2vec    = f1_score(y_test, preds_ts2vec, average='macro',    zero_division=0)

print(f"F1-Score (Weighted) TS2Vec : {f1_weighted_ts2vec * 100:.2f}%")
print(f"F1-Score (Macro) TS2Vec    : {f1_macro_ts2vec * 100:.2f}%")

Training Time  : 57.17 sec
Inference Time : 0.19 sec
F1-Score (Weighted) TS2Vec : 48.43%
F1-Score (Macro) TS2Vec    : 34.62%


## Comparaison Table

In [24]:
data = {
    "Models": [
        "ROCKET",
        "MultiROCKET",
        "MANTIS (Head / LinearBing)",
        "MANTIS (Full Fine-Tuning)",
        "Chronos (Features + LR)",
        "TimesFM (Features + LR)",
        "TS2Vec (Features + LR)",
    ],
    "Training time (s)": [
        time_train_rocket,
        time_train_multirocket,
        time_train_head,
        time_train_full,
        time_train_chronos,
        time_train_timesfm,
        time_train_ts2vec,
    ],
    "Inference time (s)": [
        time_test_rocket,
        time_test_multirocket,
        time_test_head,
        time_test_full,
        time_test_chronos,
        time_test_timesfm,
        time_test_ts2vec,
    ],
    "F1-Score Macro (%)": [
        f1_macro_rocket * 100,
        f1_macro_multirocket * 100,
        f1_macro_head * 100,
        f1_macro_full * 100,
        f1_chronos * 100,
        f1_macro_TimesFM * 100,
        f1_macro_ts2vec * 100,
    ],
    "F1-Score Weighted (%)": [
        f1_weighted_rocket * 100,
        f1_weighted_multirocket * 100,
        f1_weighted_head * 100,
        f1_weighted_full * 100,
        None,
        f1_weighted_TimesFM * 100,
        f1_weighted_ts2vec * 100,
    ]
}

df_results = pd.DataFrame(data)
df_results = df_results.round(2)
df_results

,Models,Training time (s),Inference time (s),F1-Score Macro (%),F1-Score Weighted (%)
0,ROCKET,167.41,60.22,40.07,60.28
1,MultiROCKET,110.21,9.92,63.84,53.06
2,MANTIS (Head / LinearBing),8.30,4.00,51.15,63.54
3,MANTIS (Full Fine-Tuning),682.10,4.93,56.97,68.37
4,Chronos (Features + LR),15.34,5.00,50.16,NaN
5,TimesFM (Features + LR),24.73,6.28,36.10,51.08
6,TS2Vec (Features + LR),57.17,0.19,34.62,48.43
